# QuantJourney SDK - USD Liquidity Cross-Asset Mosaic

This notebook demonstrates a QuantJourney SDK workflow that combines Fed liquidity series, rates, volatility, equity, duration, gold, dollar and crypto prices into one data mosaic.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
macro_calls = {'Fed balance sheet assets (FRED:WALCL)': qj.fred.get_fred_data_series_by_id(series_id='WALCL', start='2020-01-01'), 'Treasury General Account (FRED:WTREGEN)': qj.fred.get_fred_data_series_by_id(series_id='WTREGEN', start='2020-01-01'), 'Reverse repo operations (FRED:RRPONTSYD)': qj.fred.get_fred_data_series_by_id(series_id='RRPONTSYD', start='2020-01-01'), 'Effective Fed Funds Rate (FRED:FEDFUNDS)': qj.fred.get_effective_federal_funds_rate(start_date='2020-01-01'), '10-Year Treasury Yield (FRED:DGS10)': qj.fred.get_treasury_10y(start_date='2020-01-01'), 'Treasury repo rate': qj.fred.get_treasury_repo_rate(start_date='2020-01-01')}
vix_raw = qj.cboe.get_vix_data(start_date='2020-01-01', end_date=END)
prices, volumes = price_panel(['SPY', 'TLT', 'GLD', 'UUP'], start='2020-01-01', end=END)
btc_raw = qj.ccxt.get_historical_prices(symbol='BTC/USDT', exchange='binance', timeframe='1d', since='2020-01-01')


In [ ]:
def series_from_payload(name: str, payload: Any) -> pd.Series:
    frame = pd.DataFrame(as_rows(payload))
    if frame.empty:
        return pd.Series(dtype=float, name=name)
    date_col = next((col for col in frame.columns if 'date' in str(col).lower() or str(col).lower() in {'observation_date', 'time'}), frame.columns[0])
    numeric_cols = frame.select_dtypes(include='number').columns.tolist()
    value_col = 'value' if 'value' in frame.columns else 'close' if 'close' in frame.columns else numeric_cols[-1] if numeric_cols else None
    if value_col is None:
        return pd.Series(dtype=float, name=name)
    frame['date'] = pd.to_datetime(frame[date_col], errors='coerce')
    frame[name] = pd.to_numeric(frame[value_col], errors='coerce')
    return frame.dropna(subset=['date', name]).set_index('date')[name].sort_index()
macro = pd.concat([series_from_payload(name, payload) for name, payload in macro_calls.items()], axis=1).dropna(how='all')
vix = series_from_payload('CBOE Volatility Index (VIX)', vix_raw)
btc = series_from_payload('Bitcoin spot (CCXT:BTC/USDT)', btc_raw)
asset_prices = prices.join(btc, how='outer').dropna(how='all')


In [ ]:
if macro.empty or asset_prices.empty:
    raise RuntimeError('Liquidity mosaic requires macro observations and market prices')
liquidity_index = (macro.filter(like='Fed balance').ffill().iloc[:, 0].pct_change(26) - macro.filter(like='Treasury General').ffill().iloc[:, 0].pct_change(26).reindex(macro.index) - macro.filter(like='Reverse repo').ffill().iloc[:, 0].pct_change(26).reindex(macro.index)).rename('USD liquidity impulse')
cross_asset = asset_prices.pct_change(63).iloc[-1].rename('63d_return')
risk_state = pd.concat([liquidity_index, vix.reindex(liquidity_index.index).ffill()], axis=1).dropna()
display(macro.tail())
display(cross_asset.sort_values(ascending=False))
risk_state.tail(260).plot(title='USD liquidity impulse and VIX')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.